In [1]:
import pandas as pd

# Data
df=pd.read_csv("/home/jolivera/Documents/CloudSkin/Time-Series-Library/dataset/comparison_of_approaches/check_per_exp_avg.csv", sep=";")

# cloud_cluster_only;0;0.0;0.0;0.0;0.0;0.0;91,764705882;0.0;0.0;28750.0;0.0;99.68


In [2]:
df

,approach,num_experiments,avg_num_steps,avg_overall_mean_latency,avg_peak_latency,avg_num_times_crossed_above_sla,avg_total_seconds_above_sla_edge_cluster,avg_total_seconds_above_sla_cloud_cluster,avg_number_migrations_edge_to_cloud,avg_number_migrations_cloud_to_edge,avg_total_seconds_in_cloud_cluster,avg_total_seconds_in_edge_cluster,avg_percent_time_below_sla
0,informer,2,953.500000,0.060666,0.615010,4.500000,825.0,0.0,2.00,1.000000,16680.0,12075.0,97.131287
1,random_forest,15,942.266667,0.063393,0.450828,1.933333,328.0,32.0,6.20,6.200000,18352.0,10402.0,98.747949
2,informer_30m,4,953.750000,0.065030,0.594809,7.750000,690.0,7.5,1.25,1.250000,23557.5,5182.5,97.572958
3,informer_filtered_SLAloss,2,954.000000,0.072233,0.636597,8.500000,1170.0,45.0,3.00,3.500000,15690.0,13050.0,95.772443
4,reactive,25,939.400000,0.073439,0.528360,6.200000,1088.4,157.2,3.36,3.160000,15956.4,12801.6,95.668618
5,multiple_linear_regression,3,951.000000,0.082383,0.599166,7.000000,1130.0,50.0,2.00,1.666667,17120.0,11640.0,95.896836
6,linear_regression,3,912.333333,0.164370,0.634490,24.666667,7620.0,460.0,22.00,22.000000,10130.0,18610.0,71.885873
7,cloud_cluster_only,0,0.000000,0.000000,0.000000,0.000000,0.0,"91,764705882",0.00,0.000000,28750.0,0.0,99.680000


In [3]:
df.iloc[7,10]=24*3600 # Setting Cloud only time to 24 hours

In [4]:
# Constants
aws_price_per_hour = 0.688*0.86
edge_energy_price = 0.137  # €/kWh
edge_high_power_kw = 0.182  # kW
edge_low_power_kw = 0.119  # kW

# Cost components
df["cloud_cost"] = (df["avg_total_seconds_in_cloud_cluster"] / 3600) * aws_price_per_hour
df["edge_cost"] = (df["avg_total_seconds_in_edge_cluster"] / 3600) * (edge_energy_price * edge_high_power_kw) + ((24*3600-(df["avg_total_seconds_in_edge_cluster"]+df["avg_total_seconds_in_cloud_cluster"]))/3600)*(edge_energy_price * edge_low_power_kw)
df["edge_to_cloud_migration_cost"] = (df["avg_number_migrations_edge_to_cloud"] * aws_price_per_hour)*180 / 3600
df["cloud_to_edge_migration_cost"] = (df["avg_number_migrations_cloud_to_edge"] * edge_energy_price * edge_high_power_kw)*30 / 3600

# Total cost
df["total_cost"] = df["cloud_cost"] + df["edge_cost"] + df["edge_to_cloud_migration_cost"] + df["cloud_to_edge_migration_cost"]

# Final columns
df_final_daily = df[
    [
        "approach",
        "avg_percent_time_below_sla",
        "total_cost",
        "cloud_cost",
        "edge_cost",
        "avg_number_migrations_edge_to_cloud",
        "edge_to_cloud_migration_cost",
        "avg_number_migrations_cloud_to_edge",
        "cloud_to_edge_migration_cost"

    ]
]


# Option 1: 
They already have infrastructure, we just add the main computing instance.

In [5]:
df_final_daily

,approach,avg_percent_time_below_sla,total_cost,cloud_cost,edge_cost,avg_number_migrations_edge_to_cloud,edge_to_cloud_migration_cost,avg_number_migrations_cloud_to_edge,cloud_to_edge_migration_cost
0,informer,97.131287,3.145511,2.741451,0.344685,2.00,0.059168,1.000000,0.000208
1,random_forest,98.747949,3.534064,3.016253,0.333102,6.20,0.183421,6.200000,0.001288
2,informer_30m,97.572958,4.206060,3.871806,0.297014,1.25,0.036980,1.250000,0.000260
3,informer_filtered_SLAloss,95.772443,3.019723,2.578739,0.351505,3.00,0.088752,3.500000,0.000727
4,reactive,95.668618,3.072285,2.622523,0.349704,3.36,0.099402,3.160000,0.000657
5,multiple_linear_regression,95.896836,3.214930,2.813767,0.341649,2.00,0.059168,1.666667,0.000346
6,linear_regression,71.885873,2.710356,1.664922,0.390015,22.00,0.650848,22.000000,0.004571
7,cloud_cluster_only,99.680000,14.200320,14.200320,0.000000,0.00,0.000000,0.000000,0.000000


# Option 2:
We deploy everything.

In [6]:
ebs_cost= 0.088*0.86 # €/GB-month. Includes 3000 IOPS and 125 MB/s of throughput.
storage_needed= 100 # GB
control_plane_instance_cost= 0.0368*0.86 # t4g.medium €/hour

df["baseline_cost"] = ebs_cost*storage_needed + (24 * control_plane_instance_cost)
df["total_cost"] = df["baseline_cost"] + df["cloud_cost"] + df["edge_cost"] + df["edge_to_cloud_migration_cost"] + df["cloud_to_edge_migration_cost"]


# Final columns
df_deployment_daily = df[
    [
        "approach",
        "avg_percent_time_below_sla",
        "total_cost",
        "baseline_cost",
        "cloud_cost",
        "edge_cost",
        "avg_number_migrations_edge_to_cloud",
        "edge_to_cloud_migration_cost",
        "avg_number_migrations_cloud_to_edge",
        "cloud_to_edge_migration_cost"

    ]
]


In [7]:
df_deployment_daily

,approach,avg_percent_time_below_sla,total_cost,baseline_cost,cloud_cost,edge_cost,avg_number_migrations_edge_to_cloud,edge_to_cloud_migration_cost,avg_number_migrations_cloud_to_edge,cloud_to_edge_migration_cost
0,informer,97.131287,11.473063,8.327552,2.741451,0.344685,2.00,0.059168,1.000000,0.000208
1,random_forest,98.747949,11.861616,8.327552,3.016253,0.333102,6.20,0.183421,6.200000,0.001288
2,informer_30m,97.572958,12.533612,8.327552,3.871806,0.297014,1.25,0.036980,1.250000,0.000260
3,informer_filtered_SLAloss,95.772443,11.347275,8.327552,2.578739,0.351505,3.00,0.088752,3.500000,0.000727
4,reactive,95.668618,11.399837,8.327552,2.622523,0.349704,3.36,0.099402,3.160000,0.000657
5,multiple_linear_regression,95.896836,11.542482,8.327552,2.813767,0.341649,2.00,0.059168,1.666667,0.000346
6,linear_regression,71.885873,11.037908,8.327552,1.664922,0.390015,22.00,0.650848,22.000000,0.004571
7,cloud_cluster_only,99.680000,22.527872,8.327552,14.200320,0.000000,0.00,0.000000,0.000000,0.000000


In [8]:
df_deployment_daily[df_deployment_daily["approach"].isin(["cloud_cluster_only","reactive","random_forest"])]






,approach,avg_percent_time_below_sla,total_cost,baseline_cost,cloud_cost,edge_cost,avg_number_migrations_edge_to_cloud,edge_to_cloud_migration_cost,avg_number_migrations_cloud_to_edge,cloud_to_edge_migration_cost
1,random_forest,98.747949,11.861616,8.327552,3.016253,0.333102,6.20,0.183421,6.20,0.001288
4,reactive,95.668618,11.399837,8.327552,2.622523,0.349704,3.36,0.099402,3.16,0.000657
7,cloud_cluster_only,99.680000,22.527872,8.327552,14.200320,0.000000,0.00,0.000000,0.00,0.000000
